In [33]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

In [34]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [35]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [36]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [37]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [38]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [39]:
def tokenize_lematize(tekst):
    tokens = word_tokenize(tekst.lower())  # vse besede pišemo z malo začetnico
    tagged = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word.isalpha()    # znebimo se ločil, st,...
    ]

In [40]:
import random
from sklearn.feature_extraction import text

In [41]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()

custom_words = {
    'book', 'novel', 'story', 'reader',
    'edition', 'classic', 'introduction',
    'publish', 'note', 'cover', 'series',
    'time', 'year', 'new', 'make', 'tell',
    'begin', 'just', 'work', 'face',
    'place', 'mean', 'text', 'author', 'original', 'u', 'seller',
    'masterpiece', 'literature', 'best', 'read', 'man', 'men', 'woman', 'life',
    'fiction', 'tale', 'want', 'come', 'know'
}

all_stopwords = list(base_stopwords.union(custom_words))

In [42]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\najbolj_popularne\najboljse_knjige_opisi\*.txt')[:250]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words=all_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=1, 
                            max_df=0.9)

In [43]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['far'] not in stop_words.
  warnings.warn(


In [44]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 13575 stored elements and shape (250, 5544)>
  Coords	Values
  (0, 5445)	2
  (0, 1763)	1
  (0, 1923)	1
  (0, 2925)	1
  (0, 751)	1
  (0, 1185)	3
  (0, 2367)	2
  (0, 1999)	3
  (0, 4222)	1
  (0, 3373)	1
  (0, 149)	1
  (0, 2860)	1
  (0, 3300)	1
  (0, 3515)	1
  (0, 4444)	1
  (0, 684)	2
  (0, 4795)	1
  (0, 3476)	1
  (0, 1368)	2
  (0, 2201)	1
  (0, 1098)	1
  (0, 2877)	1
  (0, 1902)	1
  (0, 4378)	1
  (0, 553)	1
  :	:
  (249, 3174)	1
  (249, 1345)	1
  (249, 199)	1
  (249, 3552)	1
  (249, 1766)	1
  (249, 4330)	1
  (249, 1335)	1
  (249, 976)	1
  (249, 5089)	1
  (249, 2824)	1
  (249, 513)	1
  (249, 4566)	1
  (249, 1336)	1
  (249, 3274)	1
  (249, 4450)	1
  (249, 95)	1
  (249, 1018)	1
  (249, 1519)	1
  (249, 791)	1
  (249, 4875)	1
  (249, 1494)	1
  (249, 2879)	1
  (249, 837)	2
  (249, 2691)	2
  (249, 2759)	1
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
   aaron  abag

In [45]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [50]:
# test 

W, H, errors = nmf(X, 5)
print(errors)
#print(W)
#print(H)

[np.float64(142.34486395858085), np.float64(141.83507886697237), np.float64(141.29234761268265), np.float64(140.64384856857728), np.float64(140.0677661280128), np.float64(139.62683842329662), np.float64(139.24491638483093), np.float64(138.88251737162386), np.float64(138.6130894583345), np.float64(138.42247693340673), np.float64(138.28243280692453), np.float64(138.18012415004551), np.float64(138.10678157270692), np.float64(138.05715501353112), np.float64(138.02354937345842), np.float64(137.9998933119294), np.float64(137.98272426243108), np.float64(137.9700771829585), np.float64(137.95985115861737), np.float64(137.95105196884194), np.float64(137.94361985967723), np.float64(137.93720501011134), np.float64(137.93142263312984), np.float64(137.9261654428786), np.float64(137.9213299720362), np.float64(137.91639200400553), np.float64(137.91135668364984), np.float64(137.90655979075376), np.float64(137.9018922597022), np.float64(137.8971902624794), np.float64(137.89248210633468), np.float64(137.

In [51]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-15:]]
    print(f"Tema {topic_idx+1}: {' '.join(top_words)}")

Tema 1: change shadowhunters family girl save heart childhood sister daughter past mother tessa frank secret love
Tema 2: old land american set character family write people girl war great include young love world
Tema 3: blood princess hathaway dead dragomir world academy protect vampire dimitri friend strigoi love lissa rise
Tema 4: bear fall island room seven leave devon chop hang rhyme guest sit murder boy little
Tema 5: death seal letter strong tournament turn student wizard dark school lord potter voldemort hogwarts harry
